# WD Tagger API Server on Colab

通过 ngrok 暴露 onnxruntime GPU 推理 API。
选一个模型，运行后你会得到一个外网可访问的 API 地址。

In [ ]:
# @title 1. 安装依赖
!pip install -q onnxruntime-gpu fastapi uvicorn pyngrok pillow numpy
!ngrok authtoken 3EXE7ZUPVPBWdvWXR45hhfvk0HX_4a7ziRuM1USJ1CMefEzQE

In [ ]:
# @title 2. 下载模型
import os, urllib.request

# @markdown 可选模型:
# @markdown - `SmilingWolf/wd-eva02-large-tagger-v3` (最大最准, ~1.2GB)
# @markdown - `SmilingWolf/wd-vit-tagger-v3` (轻量快速, ~350MB)
# @markdown - `SmilingWolf/wd-swinv2-tagger-v3` (均衡, ~500MB)
MODEL_ID = "SmilingWolf/wd-eva02-large-tagger-v3"  # @param ["SmilingWolf/wd-eva02-large-tagger-v3", "SmilingWolf/wd-vit-tagger-v3", "SmilingWolf/wd-swinv2-tagger-v3"]

BASE = f"https://huggingface.co/{MODEL_ID}/resolve/main"
os.makedirs("/content/model", exist_ok=True)

for f in ["model.onnx", "selected_tags.csv", "config.json"]:
    path = f"/content/model/{f}"
    if not os.path.exists(path):
        print(f"Downloading {f}...")
        urllib.request.urlretrieve(f"{BASE}/{f}", path)
        print(f"  OK ({os.path.getsize(path)/1024/1024:.0f} MB)")
    else:
        print(f"{f} 已存在")

print("\n模型就绪")

In [ ]:
# @title 3. 启动 API Server
import io, json, csv, time, urllib.request, numpy as np
from PIL import Image
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
import onnxruntime as ort

# ── 加载模型 ──
session = ort.InferenceSession("/content/model/model.onnx", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

# ── 加载标签 ──
tags = []
rating_ids = [9999999, 9999998, 9999997, 9999996]  # general, sensitive, questionable, explicit
with open("/content/model/selected_tags.csv") as f:
    for row in csv.DictReader(f):
        tags.append({"id": int(row["tag_id"]), "name": row["name"], "category": int(row["category"])})

tag_names = [t["name"] for t in tags]

# ── 图片预处理 ──
def preprocess(img: Image.Image, target_size=448):
    # 居中裁剪后缩放到 target_size
    w, h = img.size
    crop = min(w, h)
    left = (w - crop) // 2
    top = (h - crop) // 2
    img = img.crop((left, top, left + crop, top + crop))
    img = img.resize((target_size, target_size), Image.BICUBIC)
    arr = np.array(img, dtype=np.float32) / 127.5 - 1.0  # [0,255] → [-1,1]
    arr = np.transpose(arr, (2, 0, 1))  # HWC → CHW
    return np.expand_dims(arr, axis=0)  # → NCHW

# ── 推理 ──
def predict(img: Image.Image, gen_threshold=0.35, char_threshold=0.35):
    t0 = time.time()
    x = preprocess(img)
    logits = session.run(None, {input_name: x})[0][0]  # (10861,)
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    elapsed = time.time() - t0

    # 评级 (rating)
    rating_confs = [(tag_names[i], float(probs[i])) for i in rating_ids]
    rating_label = max(rating_confs, key=lambda x: x[1])[0]

    # 标签过滤
    result_tags = []
    for i, prob in enumerate(probs):
        tag = tags[i]
        if tag["id"] in rating_ids:
            continue
        threshold = char_threshold if tag["category"] == 4 else gen_threshold
        if prob >= threshold:
            result_tags.append({"name": tag["name"], "category": tag["category"], "confidence": round(float(prob), 4)})

    return {
        "rating": {"label": rating_label, "confidences": [{"label": l, "confidence": c} for l, c in rating_confs]},
        "tags": [(t["name"], t["confidence"]) for t in result_tags],
        "time": round(elapsed, 2)
    }

# ── FastAPI ──
app = FastAPI(title="WD Tagger")

@app.get("/")
def root():
    return {"status": "ok", "model": MODEL_ID, "tags_count": len(tags)}

@app.post("/tag")
async def tag(file: UploadFile = File(...), gen_threshold: float = 0.35, char_threshold: float = 0.35):
    img = Image.open(io.BytesIO(await file.read())).convert("RGB")
    return predict(img, gen_threshold, char_threshold)

@app.post("/tag_url")
class TagURLRequest(BaseModel):
    url: str
    gen_threshold: float = 0.35
    char_threshold: float = 0.35

@app.post("/tag_url")
async def tag_url(req: TagURLRequest):
    resp = urllib.request.urlopen(req.url)
    img = Image.open(io.BytesIO(resp.read())).convert("RGB")
    return predict(img, req.gen_threshold, req.char_threshold)

# ── 启动 uvicorn ──
import threading, uvicorn
from pyngrok import ngrok

tunnel = ngrok.connect(8000, bind_tls=True)
print(f"\n🚀 API 地址: {tunnel.public_url}")
print(f"    POST {{url}}/tag      - 上传图片")
print(f"    POST {{url}}/tag_url  - 传图片 URL")
print(f"    GET  {{url}}/         - 健康检查\n")

uvicorn.run(app, host="0.0.0.0", port=8000)